# Corbel — YOLOv8n floor plan detection training

Trains the wall/door/window detector used by Corbel's browser-side plan import
(`corbel-detect.onnx`). Classes: 0=wall, 1=door, 2=window. Dataset: CubiCasa5k
converted to YOLO boxes (4200 train / 400 val / 400 test images).

## Before you run this

1. **Set the GPU runtime**: Runtime → Change runtime type → T4 GPU (free tier).
2. **Upload the dataset to your Google Drive root** (`MyDrive`, not a
   subfolder): `ml/corbel-dataset.zip` (1.8 GB) from this repo. Do this once —
   Drive persists it across sessions, so you won't need to re-upload for a
   resumed or repeat run.
3. Run the cells in order (Runtime → Run all works too).

Training runs ~150 epochs with early stopping (`patience=30`, so it may finish
sooner if validation loss plateaus). On a free T4 this is roughly **6–12
hours** — Colab free-tier sessions can disconnect before that; checkpoints
save to Drive every epoch (cell 5), so if it disconnects, re-open this
notebook, set `RESUME = True` in cell 5, and run cells 1–2 then 5 onward again.

Epoch count was raised from an original 50 after a 2026-09-05 benchmark showed
the previously-shipped model's boxes were imprecise in a way no amount of
confidence/NMS threshold tuning or box-merging post-processing could fix —
pointing at undertraining rather than a labeling or metric bug (see
`corbel/benchmark_out/comparison-40/` and `corbel/context.md` in the repo for
the full writeup). This is a reasoned starting point, not a verified
hyperparameter search.</cell id="987e651b">


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# 1 — GPU check (should show a Tesla T4; if this errors, set Runtime > Change
# runtime type > T4 GPU first, then Runtime > Restart session)
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [6]:
# 2 — mount Drive, unpack dataset
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/fyp-dataset/corbel-dataset.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 3 — install ultralytics
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.0 MB/s eta 0:00:00


In [7]:
# 4 — point dataset yaml at Colab path (yaml carries an absolute local-machine path)
!sed -i 's|^path:.*|path: /content/dataset|' /content/dataset/cubicasa.yaml
!cat /content/dataset/cubicasa.yaml

path: /content/dataset
train: images/train
val: images/val
test: images/test
names:
  0: wall
  1: door
  2: window


In [ ]:
from ultralytics import YOLO

RESUME = False  # set True to continue an interrupted run

if RESUME:
    model = YOLO('/content/drive/MyDrive/corbel-runs/corbel/weights/last.pt')
    model.train(resume=True)
else:
    model = YOLO('yolov8n.pt')
    model.train(
        data='/content/dataset/cubicasa.yaml',
        epochs=150,
        patience=30,
        imgsz=640,
        save_period=1,
        project='/content/drive/MyDrive/corbel-runs',
        name='corbel',
        exist_ok=True,
        optimizer='AdamW', # Use AdamW optimizer for potentially better performance
        batch=-1 # Automatically determine the optimal batch size for the GPU
    )

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.141 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/cubicasa.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, fo

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90344064 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


train: Scanning /content/dataset/labels/train... 4200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4200/4200 561.7it/s 7.5s
train: /content/dataset/images/train/high_quality_13325.png: 1 duplicate labels removed
train: /content/dataset/images/train/high_quality_14593.png: 1 duplicate labels removed
train: /content/dataset/images/train/high_quality_14692.png: 1 duplicate labels removed
train: /content/dataset/images/train/high_quality_architectural_10162.png: 1 duplicate labels removed
train: /content/dataset/images/train/high_quality_architectural_10345.png: 1 duplicate labels removed
train: /content/dataset/images/train/high_quality_architectural_10516.png: 1 duplicate labels removed
train: /content/dataset/images/train/high_quality_architectural_10589.png: 3 duplicate labels removed
train: /content/dataset/images/train/high_quality_architectural_11930.png: 1 duplicate labels removed
train: /content/dataset/images/train/high_quality_architectural_1227.png: 1 duplicate labels re

In [8]:
# 6 — evaluate on held-out test split (mAP / precision / recall per class)
from ultralytics import YOLO

best = YOLO('/content/drive/MyDrive/corbel-runs/corbel/weights/best.pt')
metrics = best.val(data='/content/dataset/cubicasa.yaml', split='test')
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
print('per-class mAP50:', dict(zip(['wall', 'door', 'window'], metrics.box.maps.tolist())))

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 48.0±28.1 MB/s, size: 335.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/dataset/labels/test... 400 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 400/400 273.5it/s 1.5s
val: /content/dataset/images/test/colorful_11399.png: 1 duplicate labels removed
val: /content/dataset/images/test/high_quality_architectural_9223.png: 1 duplicate labels removed
val: New cache created: /content/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 1.2it/s 21.2s
                   all        400      18424      0.847       0.75      0.807      0.467
                  w

In [9]:
# 7 — export ONNX for browser inference (onnxruntime-web)
onnx_path = best.export(format='onnx', imgsz=640, opset=17)
!cp {onnx_path} /content/drive/MyDrive/corbel-runs/corbel-detect.onnx
print('Saved to Drive: corbel-runs/corbel-detect.onnx')
print('Then locally:  cp corbel-detect.onnx corbel/public/models/corbel-detect.onnx')

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/corbel-runs/corbel/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (23.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 334ms
Prepared 4 packages in 1.97s
Installed 4 packages in 405ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.29.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 3.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 17...
ONNX

## Bring the model back to Corbel

1. Download `corbel-detect.onnx` from your Google Drive: `corbel-runs/corbel-detect.onnx`.
2. Locally: `cp ~/Downloads/corbel-detect.onnx corbel/public/models/corbel-detect.onnx`
3. Re-run the benchmark against the **test** split (not val — val was used for
   all the threshold/post-processing tuning that preceded this training run,
   so val numbers here would be optimistic):
   ```
   cd corbel
   node --experimental-strip-types scripts/benchmark-local-yolo.ts --split test --n 400
   ```
4. Compare honestly against the pre-retrain baseline (bbox-IoU@0.50 F1):
   wall 76.7%, door 86.5%, window 87.4%. Door/window should be much closer to
   95% now; wall may still fall short — see `corbel/context.md` for why that's
   expected rather than a sign something went wrong.